# helpdesk_functions — Reference Module (Local Function-Tool Implementations)

Unlike the other files in this section, `helpdesk_functions.py` is **not a standalone demo** — it's a shared
support module with no Azure SDK calls and no `if __name__ == "__main__"` entry point. It's imported by:

- `05_IT_HelpDesk_agent.py` — indirectly, in spirit: that script declares `FunctionTool` *schemas* whose
  `name`s match the function names defined here (it does not literally `import` this module, since it only
  registers the tool contract with Foundry).
- `07_helpdesk_client.py` — directly: `from helpdesk_functions import run_local_function`. This is the file
  that actually **executes** these functions when the Foundry agent requests a tool call.

This notebook walks through each function in the module, then demonstrates calling `run_local_function`
directly, the same way `07_helpdesk_client.py` does inside its tool-call loop.

**Difficulty:** Beginner


## Prerequisites

**pip3 packages:** None beyond the Python standard library — this module has zero external imports.

**Azure resource(s) required:** None — this is pure local Python logic with no network calls.

**Environment variables:** None.

This module is deliberately "boring": it's the part of the IT help-desk demo that has nothing to do with
Azure at all, which is exactly the point — it's the client-side implementation half of the `FunctionTool`
contract whose schema half lives in `05_IT_HelpDesk_agent.py`.


## What You'll Learn

- The shape of a **dispatcher function** (`run_local_function`) that maps a tool name (string) + arguments
  (dict) to a concrete Python call — the standard pattern for handling `function_call` items returned by an
  agent
- Why keeping tool *implementations* separate from tool *schema declarations* (`05_IT_HelpDesk_agent.py`) is a
  natural way to structure function-calling code
- How this module's three functions correspond 1:1 to the three `FunctionTool` schemas registered in `05`


### `get_password_reset_steps()`

Takes no arguments, returns a fixed, hardcoded string of numbered password-reset steps. This is the
implementation behind the `get_password_reset_steps` `FunctionTool` schema in `05_IT_HelpDesk_agent.py` — when
the agent decides to call that tool, this is the code that actually runs, client-side.

- **💡 Exam tip:** Notice this function has **no `@function_tool` decorator or type-hint magic** like
  `01_openai_agent.py`'s version — because Azure AI Foundry's `FunctionTool` doesn't call your Python function
  automatically the way the OpenAI Agents SDK does. The schema declared server-side and the implementation
  called client-side are two independent things you must keep in sync yourself.
- **🔄 Alternatives:** A production system might fetch this text from a CMS or knowledge base instead of a
  hardcoded string, or return structured data (steps as a list) instead of one long sentence.


In [ ]:
def get_password_reset_steps() -> str:
    return (
        "To reset your password: "
        "1. Go to https://accounts.company.com/reset. "
        "2. Enter your company email address. "
        "3. Check your email for a reset link. The link expires in 15 minutes. "
        "4. Follow the link and choose a new password. "
        "5. If the link does not arrive, check your spam folder or contact IT."
    )


### `get_vpn_troubleshooting_steps()`

Same shape as above: no arguments, fixed string output — the implementation behind the
`get_vpn_troubleshooting_steps` tool schema.

- **💡 Exam tip:** Zero-argument function tools still need a `parameters` schema when declared to Foundry —
  `05_IT_HelpDesk_agent.py` uses `{"type": "object", "properties": {}, "required": []}` for exactly this case,
  an "empty but valid" JSON Schema object.
- **🔄 Alternatives:** None needed here structurally, but this function could be parameterized (e.g. by
  `vpn_client_version`) if the guide needed to branch by client version.


In [ ]:
def get_vpn_troubleshooting_steps() -> str:
    return (
        "VPN troubleshooting steps: "
        "1. Confirm you are connected to the internet before launching the VPN client. "
        "2. Check that your VPN client is up to date. Version 4.2 or later is required. "
        "3. Try disconnecting and reconnecting. "
        "4. If the issue persists, restart your machine and try again. "
        "5. If you are on a public network, some ports may be blocked. Try a mobile hotspot. "
        "6. Contact IT if the problem continues after these steps."
    )


### `get_software_install_guide(software_name)`

This one takes an argument, matching the `software_name` parameter declared in the `FunctionTool` schema in
`05_IT_HelpDesk_agent.py`. It normalizes the input to lowercase and looks it up in a small dict, falling back
to a generic "contact IT" message for anything unrecognized.

- **💡 Exam tip:** The function's signature here must be **argument-compatible** with the JSON Schema
  registered server-side — the model will send `{"software_name": "Slack"}` as JSON, and the caller (in
  `07_helpdesk_client.py`) does `json.loads(item.arguments)` then unpacks it with `**arguments` into this
  function, so the dict key (`software_name`) must exactly match the Python parameter name.
- **🔄 Alternatives:** A production version might query a real IT-asset/software-catalog API instead of a
  hardcoded dict, or return a structured object (URL, steps, prerequisites) instead of one string.


In [ ]:
def get_software_install_guide(software_name: str) -> str:
    guides = {
        "slack": "To install Slack, visit https://slack.com/downloads, download the installer, and sign in with your company email.",
        "zoom": "To install Zoom, visit https://zoom.us/download, download Zoom Desktop Client, and sign in using SSO.",
        "vscode": "To install VS Code, visit https://code.visualstudio.com, download the installer, and follow the setup wizard.",
    }

    name = software_name.lower()

    return guides.get(
        name,
        f"No installation guide found for '{software_name}'. Please contact IT for assistance."
    )


### `run_local_function(function_name, arguments)` — the dispatcher

This is what `07_helpdesk_client.py` actually imports and calls. It takes the `function_name` string and
`arguments` dict extracted from a Foundry agent's `function_call` output item, and routes to the matching
local function above. Unknown function names return a clear error string rather than raising an exception —
a defensive default that keeps the calling loop from crashing on an unexpected tool name.

- **💡 Exam tip:** This `if/elif`-style dispatcher is the simplest possible implementation of the "execute the
  requested tool" half of the function-calling loop. In a larger system you'd likely replace it with a `dict`
  mapping names to callables (`{"get_password_reset_steps": get_password_reset_steps, ...}`), but the contract
  — name in, result string out — is identical either way.
- **🔄 Alternatives:** A registry/decorator pattern (similar in spirit to the OpenAI Agents SDK's
  `@function_tool` from `01_openai_agent.py`, but hand-rolled) could auto-register functions instead of a
  manual `if` chain, reducing the risk of the dispatcher and the `FunctionTool` schemas drifting out of sync.


In [ ]:
def run_local_function(function_name: str, arguments: dict) -> str:
    if function_name == "get_password_reset_steps":
        return get_password_reset_steps()

    if function_name == "get_vpn_troubleshooting_steps":
        return get_vpn_troubleshooting_steps()

    if function_name == "get_software_install_guide":
        return get_software_install_guide(**arguments)

    return f"Unknown function requested: {function_name}"


### Example: calling `run_local_function` directly

Before wiring this into a full agent loop (`07_helpdesk_client.py`), it's worth confirming the dispatcher
behaves as expected on its own — exactly what an agent's `function_call` output would trigger, minus the
Azure round-trip.


In [ ]:
# Simulates what 07_helpdesk_client.py does after parsing a `function_call` item's arguments.
print(run_local_function("get_vpn_troubleshooting_steps", {}))
print()
print(run_local_function("get_software_install_guide", {"software_name": "VSCode"}))
print()
print(run_local_function("unknown_tool", {}))


## Summary

`helpdesk_functions.py` is the client-side "backend" for the IT help-desk agent's function tools: three plain
functions returning canned guidance strings, plus a `run_local_function` dispatcher that maps a tool name and
argument dict to one of them. It has no Azure dependency at all — it's pure, testable Python — which is the
whole point of keeping tool *implementation* separate from tool *schema* (`05_IT_HelpDesk_agent.py`) and tool
*execution loop* (`07_helpdesk_client.py`).


## Try It Yourself

1. **Easy:** Call `run_local_function("get_password_reset_steps", {})` and confirm the output matches the
   string in `get_password_reset_steps()`.
2. **Intermediate:** Add a new function `get_printer_setup_steps()` plus a matching branch in
   `run_local_function`, then add the corresponding `FunctionTool` schema to `05_IT_HelpDesk_agent.py` and
   keep both in sync.
3. **Advanced:** Refactor `run_local_function` to use a `dict` of `{name: callable}` instead of `if/elif`, and
   handle the "call with wrong/missing arguments" case gracefully (e.g. catching `TypeError` from `**arguments`
   unpacking) rather than letting it propagate.
